# Imports and Data Load

In [1]:
!pip install transformers

In [2]:
from transformers import BertTokenizer, TFBertModel
from transformers import RobertaTokenizer, TFRobertaModel
from transformers import XLMRobertaTokenizer, TFXLMRobertaModel
from tensorflow.keras.layers import Input, Dense, Dropout, Layer, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd

In [3]:
train_data = pd.read_csv('https://drive.google.com/uc?id=1-fZUIY8HJ_P6M8bQ2LUITwR-G_zgIDK5')
test_data = pd.read_csv('https://drive.google.com/uc?id=1k-mFYjbegv0OW5NgF1f0gMX8bAQgSL6V')

In [5]:
X_train_full = train_data['Text']
y_train_full = train_data['Sentiment']
X_test = test_data

X_train, X_val, y_train, y_val = train_test_split(
    train_data['Text'],
    train_data['Sentiment'],
    test_size=0.15,
    random_state=42,
    stratify=train_data['Sentiment']
)

# Load BERT, RoBERTa, and XLM-R

In [6]:
bert_name = "bert-base-uncased"
bert_tokenizer = BertTokenizer.from_pretrained(bert_name)
bert_model = TFBertModel.from_pretrained(bert_name, from_pt=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions

In [7]:
roberta_name = "roberta-base"
roberta_tokenizer = RobertaTokenizer.from_pretrained(roberta_name)
roberta_model = TFRobertaModel.from_pretrained(roberta_name, from_pt=True)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaModel: ['lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.decoder.weight']
- This IS expected if you are initializing TFRobertaModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFRobertaModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


In [8]:
xlm_r_name = "xlm-roberta-base"
xlm_r_tokenizer = XLMRobertaTokenizer.from_pretrained(xlm_r_name)
xlm_r_model = TFXLMRobertaModel.from_pretrained(xlm_r_name, from_pt=True)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFXLMRobertaModel: ['lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.dense.bias', 'lm_head.decoder.weight']
- This IS expected if you are initializing TFXLMRobertaModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFXLMRobertaModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFXLMRobertaModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFXLMRobertaModel for predictions without further training.


# Preprocess

### Encode text

In [9]:
def encode_texts(texts, tokenizer, max_len=128):
    return tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='tf'
    )

### Wrapper layer

In [10]:
class WrapLayer(Layer):
    def __init__(self, bert_model, **kwargs):
        super(WrapLayer, self).__init__(**kwargs)
        self.bert = bert_model

    def call(self, inputs):
        input_ids, attention_mask = inputs
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, 0, :]

# Compile BERT, RoBERTa, and XLM-R

### BERT

In [25]:
X_train_enc_bert = encode_texts(X_train, bert_tokenizer)
X_val_enc_bert = encode_texts(X_val, bert_tokenizer)

label_enc = LabelEncoder()
y_train_enc_bert = label_enc.fit_transform(y_train)
y_val_enc_bert = label_enc.transform(y_val)
num_classes = len(label_enc.classes_)

input_ids = Input(shape=(128,), dtype='int32', name='input_ids')
attention_mask = Input(shape=(128,), dtype='int32', name='attention_mask')

bert_layer = WrapLayer(bert_model)
cls_token = bert_layer([input_ids, attention_mask])

x = Dense(128, activation='relu')(cls_token)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
output = Dense(num_classes, activation='softmax')(x)

bert_tf_model = Model(inputs=[input_ids, attention_mask], outputs=output)

bert_tf_model.compile(
    optimizer=Adam(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

### RoBERTa

In [23]:
X_train_enc_roberta = encode_texts(X_train, roberta_tokenizer)
X_val_enc_roberta   = encode_texts(X_val, roberta_tokenizer)

label_enc = LabelEncoder()
y_train_enc_roberta = label_enc.fit_transform(y_train)
y_val_enc_roberta   = label_enc.transform(y_val)
num_classes = len(label_enc.classes_)

input_ids = Input(shape=(128,), dtype='int32', name='input_ids')
attention_mask = Input(shape=(128,), dtype='int32', name='attention_mask')

roberta_layer = WrapLayer(roberta_model)
cls_token = roberta_layer([input_ids, attention_mask])

x = Dense(128, activation='relu')(cls_token)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
output = Dense(num_classes, activation='softmax')(x)

roberta_tf_model = Model(inputs=[input_ids, attention_mask], outputs=output)

roberta_tf_model.compile(
    optimizer=Adam(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

### XLM-R

In [13]:
X_train_enc_xlm_r = encode_texts(X_train, xlm_r_tokenizer)
X_val_enc_xlm_r   = encode_texts(X_val, xlm_r_tokenizer)

label_enc = LabelEncoder()
y_train_enc_xlm_r = label_enc.fit_transform(y_train)
y_val_enc_xlm_r   = label_enc.transform(y_val)
num_classes = len(label_enc.classes_)

input_ids = Input(shape=(128,), dtype='int32', name='input_ids')
attention_mask = Input(shape=(128,), dtype='int32', name='attention_mask')

xlm_r_layer = WrapLayer(xlm_r_model)
cls_token = xlm_r_layer([input_ids, attention_mask])

x = Dense(128, activation='relu')(cls_token)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)
output = Dense(num_classes, activation='softmax')(x)

xlm_r_tf_model = Model(inputs=[input_ids, attention_mask], outputs=output)

xlm_r_tf_model.compile(
    optimizer=Adam(learning_rate=2e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Analysis

### Tokenizer Comparison

In [14]:
random_indices = np.random.choice(len(train_data), size=3, replace=False)
sample_texts = train_data.iloc[random_indices]['Text'].tolist()

WordPiece (BERT)

In [15]:
for i in range (3):
    print(f"Text {i + 1}:")
    text = sample_texts[i]
    print(f"Original:\n{text}")
    bert_tokens = bert_tokenizer.tokenize(text)
    print(f"WordPiece tokens:\n{bert_tokens}\n")

Text 1:
Original:
Contentment in the simplicity of a quiet Sunday.
WordPiece tokens:
['content', '##ment', 'in', 'the', 'simplicity', 'of', 'a', 'quiet', 'sunday', '.']

Text 2:
Original:
Nostalgic memories flood in while revisiting childhood favorites.
WordPiece tokens:
['nos', '##tal', '##gic', 'memories', 'flood', 'in', 'while', 'rev', '##isi', '##ting', 'childhood', 'favorites', '.']

Text 3:
Original:
Desperation whispers, the silent plea for a glimmer of hope in the abyss.
WordPiece tokens:
['desperation', 'whispers', ',', 'the', 'silent', 'plea', 'for', 'a', 'g', '##lim', '##mer', 'of', 'hope', 'in', 'the', 'abyss', '.']



BPE (RoBERTa)

In [16]:
for i in range (3):
    print(f"Text {i + 1}:")
    text = sample_texts[i]
    print(f"Original:\n{text}")
    roberta_tokens = roberta_tokenizer.tokenize(text)
    print(f"BPE:\n{roberta_tokens}\n")

Text 1:
Original:
Contentment in the simplicity of a quiet Sunday.
BPE:
['Content', 'ment', 'Ġin', 'Ġthe', 'Ġsimplicity', 'Ġof', 'Ġa', 'Ġquiet', 'ĠSunday', '.']

Text 2:
Original:
Nostalgic memories flood in while revisiting childhood favorites.
BPE:
['N', 'ost', 'alg', 'ic', 'Ġmemories', 'Ġflood', 'Ġin', 'Ġwhile', 'Ġrevis', 'iting', 'Ġchildhood', 'Ġfavorites', '.']

Text 3:
Original:
Desperation whispers, the silent plea for a glimmer of hope in the abyss.
BPE:
['Des', 'per', 'ation', 'Ġwhispers', ',', 'Ġthe', 'Ġsilent', 'Ġplea', 'Ġfor', 'Ġa', 'Ġgl', 'immer', 'Ġof', 'Ġhope', 'Ġin', 'Ġthe', 'Ġabyss', '.']



SentencePiece (XLM-R)

In [17]:
for i in range (3):
    print(f"Text {i + 1}:")
    text = sample_texts[i]
    print(f"Original:\n{text}")
    xlm_r_tokens = xlm_r_tokenizer.tokenize(text)
    print(f"SentencePiece:\n{xlm_r_tokens}\n")

Text 1:
Original:
Contentment in the simplicity of a quiet Sunday.
SentencePiece:
['▁Content', 'ment', '▁in', '▁the', '▁simplici', 'ty', '▁of', '▁a', '▁quiet', '▁Sunday', '.']

Text 2:
Original:
Nostalgic memories flood in while revisiting childhood favorites.
SentencePiece:
['▁No', 'stal', 'gi', 'c', '▁memorie', 's', '▁flood', '▁in', '▁while', '▁revi', 'si', 'ting', '▁child', 'hood', '▁favorite', 's', '.']

Text 3:
Original:
Desperation whispers, the silent plea for a glimmer of hope in the abyss.
SentencePiece:
['▁De', 'sper', 'ation', '▁w', 'his', 'pers', ',', '▁the', '▁silent', '▁ple', 'a', '▁for', '▁a', '▁gli', 'mmer', '▁of', '▁hope', '▁in', '▁the', '▁aby', 's', 's', '.']



### Model Comparison

BERT

In [26]:
class_weights_bert = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_enc_bert),
    y=y_train_enc_bert
)
class_weight_dict_bert = dict(enumerate(class_weights_bert))

bert_history = bert_tf_model.fit(
    {'input_ids': X_train_enc_bert['input_ids'], 'attention_mask': X_train_enc_bert['attention_mask']},
    y_train_enc_bert,
    validation_data=(
        {'input_ids': X_val_enc_bert['input_ids'], 'attention_mask': X_val_enc_bert['attention_mask']},
        y_val_enc_bert
    ),
    epochs=8,
    batch_size=32,
    class_weight=class_weight_dict_bert
)

val_loss, val_acc = bert_tf_model.evaluate(
    {'input_ids': X_val_enc_bert['input_ids'], 'attention_mask': X_val_enc_bert['attention_mask']},
    y_val_enc_bert
)
print("BERT Validation Accuracy:", val_acc)

Epoch 1/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.4601 - loss: 1.1297 - val_accuracy: 0.8409 - val_loss: 0.4747
Epoch 2/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 282ms/step - accuracy: 0.4877 - loss: 0.9873 - val_accuracy: 0.8409 - val_loss: 0.4622
Epoch 3/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 284ms/step - accuracy: 0.4928 - loss: 0.9564 - val_accuracy: 0.8409 - val_loss: 0.4552
Epoch 4/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 287ms/step - accuracy: 0.4896 - loss: 0.9596 - val_accuracy: 0.8409 - val_loss: 0.4521
Epoch 5/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 290ms/step - accuracy: 0.5648 - loss: 0.8700 - val_accuracy: 0.8409 - val_loss: 0.4539
Epoch 6/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 294ms/step - accuracy: 0.5251 - loss: 0.9031 - val_accuracy: 0.8182 - val_loss: 0.4598
Epoch 7/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 291ms/step - accuracy: 0.5016 - loss: 0.7838 - val_accuracy: 0.8182 - val_loss: 0.4676
Epoch 8/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 285ms/step - accuracy: 0.5518 - loss: 0.8055 - val_accuracy: 0.8182 - val

RoBERTa

In [24]:
class_weights_roberta = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_enc_roberta),
    y=y_train_enc_roberta
)
class_weight_dict_roberta = dict(enumerate(class_weights_roberta))

roberta_history = roberta_tf_model.fit(
    {'input_ids': X_train_enc_roberta['input_ids'], 'attention_mask': X_train_enc_roberta['attention_mask']},
    y_train_enc_roberta,
    validation_data=(
        {'input_ids': X_val_enc_roberta['input_ids'], 'attention_mask': X_val_enc_roberta['attention_mask']},
        y_val_enc_roberta
    ),
    epochs=8,
    batch_size=32,
    class_weight=class_weight_dict_roberta
)

val_loss, val_acc = roberta_tf_model.evaluate(
    {'input_ids': X_val_enc_roberta['input_ids'], 'attention_mask': X_val_enc_roberta['attention_mask']},
    y_val_enc_roberta
)
print("BERT Validation Accuracy:", val_acc)

Epoch 1/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.5759 - loss: 0.8480 - val_accuracy: 0.8409 - val_loss: 0.4420
Epoch 2/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 252ms/step - accuracy: 0.6168 - loss: 0.6500 - val_accuracy: 0.8409 - val_loss: 0.4498
Epoch 3/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 253ms/step - accuracy: 0.6050 - loss: 0.6874 - val_accuracy: 0.8409 - val_loss: 0.4606
Epoch 4/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 256ms/step - accuracy: 0.5529 - loss: 0.7444 - val_accuracy: 0.8409 - val_loss: 0.4738
Epoch 5/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 260ms/step - accuracy: 0.6204 - loss: 0.6803 - val_accuracy: 0.8409 - val_loss: 0.4874
Epoch 6/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 257ms/step - accuracy: 0.5662 - loss: 0.7691 - val_accuracy: 0.8409 - val_loss: 0.5032
Epoch 7/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 254ms/step - accuracy: 0.5892 - loss: 0.6791 - val_accuracy: 0.8409 - val_loss: 0.5175
Epoch 8/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 255ms/step - accuracy: 0.5825 - loss: 0.6495 - val_accuracy: 0.8409 - val

XLM-R

In [20]:
class_weights_xlm_r = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_enc_xlm_r),
    y=y_train_enc_xlm_r
)
class_weight_dict_xlm_r = dict(enumerate(class_weights_xlm_r))

xlm_r_history = xlm_r_tf_model.fit(
    {'input_ids': X_train_enc_xlm_r['input_ids'], 'attention_mask': X_train_enc_xlm_r['attention_mask']},
    y_train_enc_xlm_r,
    validation_data=(
        {'input_ids': X_val_enc_xlm_r['input_ids'], 'attention_mask': X_val_enc_xlm_r['attention_mask']},
        y_val_enc_xlm_r
    ),
    epochs=8,
    batch_size=32,
    class_weight=class_weight_dict_xlm_r
)

val_loss, val_acc = xlm_r_tf_model.evaluate(
    {'input_ids': X_val_enc_xlm_r['input_ids'], 'attention_mask': X_val_enc_xlm_r['attention_mask']},
    y_val_enc_xlm_r
)
print("BERT Validation Accuracy:", val_acc)

Epoch 1/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 37s 1s/step - accuracy: 0.4120 - loss: 1.0955 - val_accuracy: 0.8409 - val_loss: 0.4560
Epoch 2/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 249ms/step - accuracy: 0.4532 - loss: 0.9178 - val_accuracy: 0.8409 - val_loss: 0.4649
Epoch 3/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 248ms/step - accuracy: 0.4443 - loss: 0.8328 - val_accuracy: 0.8409 - val_loss: 0.4756
Epoch 4/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 249ms/step - accuracy: 0.4763 - loss: 0.7727 - val_accuracy: 0.8409 - val_loss: 0.4873
Epoch 5/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 253ms/step - accuracy: 0.4875 - loss: 0.7532 - val_accuracy: 0.8409 - val_loss: 0.4989
Epoch 6/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 253ms/step - accuracy: 0.5123 - loss: 0.7895 - val_accuracy: 0.8409 - val_loss: 0.5104
Epoch 7/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 255ms/step - accuracy: 0.5388 - loss: 0.7562 - val_accuracy: 0.8409 - val_loss: 0.5214
Epoch 8/8
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 259ms/step - accuracy: 0.5255 - loss: 0.6867 - val_accuracy: 0.8409 - val